# Exploring the vehicle telemetry with Spark

This notebook runs its own Spark session, on the **same table catalog as dbt**: the Hive metastore in the
`metastore-db` Postgres container (see `spark/spark-defaults.conf`). So you can query the tables dbt built,
and the raw sources it reads, while `docker compose run --rm dbt build` runs in another terminal.

- The notebook's Spark UI is at http://localhost:4040 once the session below has started.
- Build the dbt models first (`docker compose run --rm dbt build`), otherwise there are no tables yet.

In [ ]:
from pyspark.sql import SparkSession, functions as F

# All settings (catalog, warehouse, spark-protobuf) come from SPARK_CONF_DIR.
spark = SparkSession.builder.appName("explore-vehicle-telemetry").enableHiveSupport().getOrCreate()
spark

In [ ]:
spark.sql("show tables in vehicle_telemetry").show(truncate=False)

## Daily summary per vehicle

The dbt model `fct_vehicle_daily`: one row per vehicle and day. `toPandas()` brings a small result to the
notebook for display and charts; keep the heavy lifting in Spark.

In [ ]:
daily = (
    spark.table("vehicle_telemetry.fct_vehicle_daily")
    .join(spark.table("vehicle_telemetry.dim_vehicles").select("vehicle_id", "vehicle_type"), "vehicle_id")
    .orderBy("event_date", "vehicle_id")
    .toPandas()
)
daily[["event_date", "vehicle_id", "vehicle_type", "reading_count", "distance_km", "avg_moving_speed_kmh",
       "max_engine_temp_c", "harsh_braking_count", "had_check_engine_light"]].round(1)

In [ ]:
import matplotlib.pyplot as plt

# Chart styling: thin marks, recessive grid, text in ink colors (never the
# series color), and a validated categorical palette (slots 1-3).
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
INK, INK_SECONDARY, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "font.family": "sans-serif", "font.size": 10, "text.color": INK,
    "axes.edgecolor": GRID, "axes.labelcolor": INK_SECONDARY, "axes.titleweight": "semibold",
    "axes.titlesize": 12, "axes.titlelocation": "left", "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 1, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
})

### Distance driven per vehicle, latest day

One series, so one color and no legend; values are labeled at the bar tips.

In [ ]:
latest = daily[daily["event_date"] == daily["event_date"].max()].sort_values("distance_km")
fig, ax = plt.subplots(figsize=(7, 0.45 * len(latest) + 1))
bars = ax.barh(latest["vehicle_id"], latest["distance_km"], height=0.5, color=SERIES[0])
ax.bar_label(bars, labels=[f"{v:,.1f} km" for v in latest["distance_km"]], padding=4, color=INK_SECONDARY)
ax.set_title(f"Distance driven on {latest['event_date'].max()}")
ax.set_xlabel("km")
ax.grid(axis="y", visible=False)
ax.set_axisbelow(True)
ax.tick_params(axis="y", length=0, labelcolor=INK)
ax.margins(x=0.15)
plt.show()

### Engine temperature of the three hottest vehicles

From the staging model, one reading per second, over the latest 15 minutes of data. The lines run close
together, so the legend identifies them rather than labels at the line ends.

In [ ]:
import matplotlib.dates as mdates

# The latest 15 minutes of data (earlier runs of the playground leave gaps).
recent = spark.table("vehicle_telemetry.stg_vehicle_telemetry")
latest_ts = recent.agg(F.max("recorded_at")).first()[0]
recent = recent.where(F.col("recorded_at") >= F.lit(latest_ts) - F.expr("INTERVAL 15 MINUTES"))

hottest = (
    recent.groupBy("vehicle_id").agg(F.max("engine_temp_c").alias("max_temp"))
    .orderBy(F.desc("max_temp")).limit(3)
    .toPandas()["vehicle_id"].tolist()
)
temps = (
    recent.where(F.col("vehicle_id").isin(hottest))
    .select("vehicle_id", "recorded_at", "engine_temp_c")
    .orderBy("recorded_at")
    .toPandas()
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.set_axisbelow(True)
ax.axhline(105, color=MUTED, linewidth=1)
ax.annotate("overheating (105 °C)", (1, 105), xycoords=("axes fraction", "data"), xytext=(0, 4),
            textcoords="offset points", ha="right", color=MUTED)
for color, vehicle in zip(SERIES, hottest):
    rows = temps[temps["vehicle_id"] == vehicle]
    ax.plot(rows["recorded_at"], rows["engine_temp_c"], color=color, linewidth=2, label=vehicle)
ax.set_title("Engine temperature, latest 15 minutes", pad=28)
ax.set_ylabel("°C")
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=3, frameon=False, borderaxespad=0.2)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
plt.show()

## The raw archive: decode the protobuf yourself

`raw_vehicle_telemetry_records` is the dbt source view over the files the protobuf consumer writes: one row
per Kafka record, with the message still serialized in `value`. Decode it with Spark's `from_protobuf` and the
descriptor set the consumer writes next to the files.

In [ ]:
from pyspark.sql.protobuf.functions import from_protobuf

DESCRIPTOR = "/data/protobuf/vehicle_telemetry/_schema/descriptor_set.desc"
records = spark.table("vehicle_telemetry.raw_vehicle_telemetry_records")
records.groupBy("message_type").count().show(truncate=False)

decoded = records.select(
    "partition", "offset",
    from_protobuf("value", "vehicle.v1.VehicleTelemetry", descFilePath=DESCRIPTOR,
                  options={"mode": "PERMISSIVE", "emit.default.values": "true"}).alias("m"),
)
(decoded.where("m is not null")
    .select("partition", "offset", "m.vehicle_id", "m.timestamp", "m.speed_kmh", "m.engine_temp_c", "m.events")
    .orderBy(F.desc("m.timestamp"))
    .show(5, truncate=False))

## Try it: run dbt at the same time

In a terminal, run `docker compose run --rm dbt build`, and rerun the cells above while it's running. Both
Spark sessions use the table catalog in Postgres, so neither blocks the other. (Before the shared catalog,
Spark kept it in an embedded Derby database that only one process could open.)

When you're done, stop the session to free its memory:

In [ ]:
spark.stop()